# Task 5 — Build a RAG System using LangChain
Loads a PDF **or** an Excel sheet as the knowledge base and answers questions using a retriever + generator pipeline.

Pipeline: `Document Loader -> Text Splitter -> Embedding -> Vector Database -> Retriever -> LLM (Generator)`

## Imports

In [12]:
!pip install -q --upgrade --force-reinstall \
    "langchain==0.3.14" "langchain-core==0.3.29" \
    "langchain-community==0.3.14" "langchain-text-splitters==0.3.5" \
    faiss-cpu sentence-transformers transformers accelerate pypdf openpyxl pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━

**Important — restart the kernel/runtime now.** Kaggle/Colab pre-import an older `langchain_core` at
startup, so even a fresh `pip install` above won't take effect until you restart: `Run > Restart session`
(Colab) or `Run > Restart Kernel` (Kaggle), then continue from the next cell **without re-running the install cell**.

In [14]:
import os
import torch
import pandas as pd

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline

ImportError: cannot import name 'hf_api' from 'transformers.utils.hub' (/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py)

## Step 1 — Document Loader
Accepts either a `.pdf` file or an `.xlsx` / `.xls` sheet and returns a unified list of LangChain `Document` objects, so the rest of the pipeline doesn't care which one was used.

**Find your file on Kaggle first.** Kaggle mounts datasets at `/kaggle/input/<dataset-folder>/...`
— not under a `datasets/` subfolder, and the folder name is auto-generated from your dataset's
title, so it may not match what you typed. Run the next cell to find the real path, then confirm
`DATA_PATH` below matches what it prints.

In [ ]:
import glob

found = glob.glob("/kaggle/input/**/*.xlsx", recursive=True) + \
        glob.glob("/kaggle/input/**/*.xls", recursive=True) + \
        glob.glob("/kaggle/input/**/*.pdf", recursive=True)

if found:
    print("Found file(s):")
    for f in found:
        print(" ", f)
else:
    print("No .pdf/.xlsx/.xls found under /kaggle/input — check the dataset is added to this notebook (Add Input, right panel).")

Copy the exact path printed above into `DATA_PATH` below (or, if there's only one match, uncomment the auto-assign line).

In [ ]:
DATA_PATH = "/kaggle/input/datasets/roqayamuhammad/langchain-concepts-sample-xlsx"  # <-- paste the exact path printed above
 DATA_PATH = found[0]  # uncomment to auto-use the first match instead

## Step 2 — Text Splitter
PDF pages are long, so they get chunked. Excel rows are already small, so chunking them further would just break single facts apart — they pass through as-is.

In [ ]:
def split_documents(documents, chunk_size=800, chunk_overlap=150):
    is_tabular = all(d.metadata.get("sheet") is not None for d in documents) if documents else False
    if is_tabular:
        return documents  # one chunk per row already

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

## Step 3 & 4 — Embedding + Vector Database
Chunks are embedded into vectors and stored in a FAISS index, saved locally so ingestion only needs to run once.

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_STORE_PATH = "faiss_index"

def build_vector_store(file_path: str, save_path: str = VECTOR_STORE_PATH):
    documents = load_documents(file_path)
    chunks = split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(save_path)

    print(f"Vector store built: {len(chunks)} chunks indexed from '{file_path}'.")
    return vector_store

In [ ]:
vector_store = build_vector_store(DATA_PATH)

## Load the Generator (LLM)

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # swap for a bigger model if you have the GPU/RAM for it

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)
print("Generator ready.")

## Step 5 — Retriever

In [ ]:
def retrieve_chunks(query: str, k: int = 3):
    return vector_store.similarity_search(query, k=k)

## Step 6 — Generator
The LLM answers using ONLY the retrieved chunks — this is what keeps RAG grounded instead of letting the model guess from its own training data.

In [ ]:
def generate_answer(query: str, context_chunks) -> str:
    context_text = "\n\n".join(doc.page_content for doc in context_chunks)

    prompt = f"""
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "Not found in the document."

Context:
{context_text}

Question:
{query}

Answer:"""

    return llm.invoke(prompt).strip()

## Full RAG Pipeline (Retriever + Generator)

In [ ]:
def ask(query: str, k: int = 3) -> str:
    chunks = retrieve_chunks(query, k=k)
    return generate_answer(query, chunks)

## Test
Replace `DATA_PATH` above with the provided file, rerun `build_vector_store`, then ask real questions about it.

In [ ]:
print(ask("What is this document about?"))

In [ ]:
questions = [
    "Summarize the main topic in two sentences.",
    "List the key points mentioned.",
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")